In [103]:
print("all ok")

all ok


In [104]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS

In [105]:
load_dotenv()

True

In [106]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

LOADING DATA

In [107]:
DATA_FILE_PATH = os.path.join("data","hr_policy.txt")

DATA INGESTION

In [108]:
loader = TextLoader(DATA_FILE_PATH,encoding="utf-8")
documents = loader.load()

print("data loaded ")
print(documents)

data loaded 
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nD

### Lang chain document 
lang chain process everything in form of documents 

Documents:  

Page content- the actual data 
```
print(documents[0].page_content)
```

Metadata - extra information about data
```
print(documents[0].metadata)
```

In [109]:
print(len(documents[0].page_content))

2597


### Splitting data

we are using recursivecharactertextsplitter : what it does is it try to chunk document in heirarchial order of
```  [ /n/n , /n , " " , "" ] ```
at each level it try to create them if they fit chunk size then okay else it moves to next splitter in heirarchy

Recursive chunking tries to preserve the natural context and semantic boundaries of the text as humans normally understand it.

In [110]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)
chunks = text_splitter.split_documents(documents)

By creating chunks of document each chunk is now treated as a document with meta data and page_content

In [111]:
print(len(chunks))

9


### Embedding for chunks

In [112]:
embeddings_model = JinaEmbeddings( model_name="jina-embeddings-v5-text-small")
print("Emb model is ready with the name:",embeddings_model.model_name)

Emb model is ready with the name: jina-embeddings-v5-text-small


Store data in vector database

In [113]:
vector_store = FAISS.from_documents(chunks,embeddings_model)
print(f'chunks are stored {vector_store.index.ntotal}')

chunks are stored 9


In [114]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



Tool for llm

In [115]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### Data retrival

In [116]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5
)

llm.model_name

'openai/gpt-oss-120b'

### AI Agent

LLM-Brain
Tool- super powers
Memory- for context (for now we are not going to add it)

In [117]:
from langchain.agents import create_agent

In [122]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt = """ 
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    If the question is too off topic outside of company do not answer it
    """
)

In [123]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [124]:
response = ask_hr_assistant("tell me which org you work for")

QUESTION: tell me which org you work for
------------------------------------------------------------
ANSWER: I’m the friendly HR assistant for **Acme Crop**. How can I help you today?

